In [ ]:
import cv2
import numpy as np
import serial
import time

# -------- SERIAL SETUP & HANDSHAKE --------
arduino = serial.Serial('COM3', 9600, timeout=1)  # change COM port
print("Waiting for Arduino to initialize...")
while True:
    if arduino.in_waiting:
        msg = arduino.readline().decode().strip()
        print(f"Arduino: {msg}")
        if msg == "SYSTEM READY":
            break
print("Handshake complete. Starting camera...")

# -------- CAMERA --------
cap = cv2.VideoCapture(0)

# -------- CONFIGURATION --------
# The X-coordinate where the robotic arm is perfectly aligned to grab
PICK_X_CENTER = 320 
TOLERANCE = 30 # How close to the center it needs to be to trigger

# -------- COLOR RANGES (HSV) --------
color_ranges = {
    "red1": ([0, 120, 70], [10, 255, 255]),
    "red2": ([170, 120, 70], [180, 255, 255]),
    "blue": ([100, 150, 50], [140, 255, 255]),
    "green": ([40, 70, 70], [80, 255, 255])
}

# -------- ORDER QUEUE --------
order_queue = ["red", "blue", "green"]

# -------- STATE --------
busy = False
last_detected = None
stable_count = 0
REQUIRED_STABLE = 5

# -------- SEND COMMAND --------
def send_command(cmd):
    global busy
    print(f"Sending: {cmd}")
    arduino.write((cmd + "\n").encode())
    busy = True

# -------- MAIN LOOP --------
while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Draw the Pick Zone target line on screen
    cv2.line(frame, (PICK_X_CENTER - TOLERANCE, 0), (PICK_X_CENTER - TOLERANCE, frame.shape[0]), (0, 255, 255), 1)
    cv2.line(frame, (PICK_X_CENTER + TOLERANCE, 0), (PICK_X_CENTER + TOLERANCE, frame.shape[0]), (0, 255, 255), 1)
    cv2.putText(frame, "PICK ZONE", (PICK_X_CENTER - 40, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 2)

    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    detected_color = None
    in_pick_zone = False

    for color_name, (lower, upper) in color_ranges.items():
        if color_name == "red2":
            continue

        lower = np.array(lower)
        upper = np.array(upper)
        mask = cv2.inRange(hsv, lower, upper)

        if color_name == "red1":
            lower2, upper2 = color_ranges["red2"]
            mask2 = cv2.inRange(hsv, np.array(lower2), np.array(upper2))
            mask = mask | mask2
            color_name = "red"

        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        for cnt in contours:
            area = cv2.contourArea(cnt)

            if area > 1500:
                x, y, w, h = cv2.boundingRect(cnt)
                cx = x + w // 2 # Center X of the object
                cy = y + h // 2 # Center Y of the object
                
                # Draw bounding box and center dot
                cv2.rectangle(frame, (x, y), (x+w, y+h), (0,255,0), 2)
                cv2.circle(frame, (cx, cy), 5, (255, 0, 0), -1)
                cv2.putText(frame, color_name, (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,0), 2)

                # Check if the object's center has crossed into the pick zone
                if abs(cx - PICK_X_CENTER) <= TOLERANCE:
                    detected_color = color_name
                    in_pick_zone = True
                
                break # Only track the largest contour of this color

    # -------- STABILITY & ZONE CHECK --------
    if detected_color == last_detected and in_pick_zone:
        stable_count += 1
    else:
        stable_count = 0

    last_detected = detected_color if in_pick_zone else None

    # -------- DECISION --------
    if (detected_color is not None and
        stable_count >= REQUIRED_STABLE and
        not busy):

        print(f"Object in zone, stable! Color: {detected_color}")

        if detected_color in order_queue:
            print("MATCH → picking up")
            # Send the generic pick command
            send_command("PICK")
            order_queue.remove(detected_color)
        else:
            print(f"Not needed: {detected_color}")

        stable_count = 0 # Reset so it doesn't double-trigger

    # -------- SERIAL FEEDBACK --------
    if arduino.in_waiting:
        msg = arduino.readline().decode().strip()
        print(f"Arduino: {msg}")

        if "ACTION: DONE" in msg:
            busy = False

    # -------- DISPLAY --------
    cv2.putText(frame, f"Orders: {order_queue}", (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)
    cv2.imshow("System", frame)

    if cv2.waitKey(1) & 0xFF == 27:
        break

# -------- CLEANUP --------
cap.release()
cv2.destroyAllWindows()
arduino.close()

In [2]:
import cv2
import numpy as np
import serial
import time

# -------- SERIAL SETUP & HANDSHAKE --------
arduino = serial.Serial('COM6', 9600, timeout=1)  # change COM port
print("Waiting for Arduino to initialize...")
while True:
    if arduino.in_waiting:
        msg = arduino.readline().decode().strip()
        print(f"Arduino: {msg}")
        if msg == "SYSTEM READY":
            break
print("Handshake complete. Starting camera...")

# -------- CAMERA --------
cap = cv2.VideoCapture(0)

# -------- CONFIGURATION --------
PICK_X_CENTER = 320    # The X-coordinate where the arm is aligned
TOLERANCE = 30         # How close to the center it needs to be
MIN_AREA = 1500        # Minimum pixel area to be considered an object
REQUIRED_STABLE = 5    # Frames needed to confirm the object

# -------- TARGET COLORS --------
# Changed from a depleting queue to a static set of valid targets
TARGET_COLORS = {"red", "blue", "green"}

color_ranges = {
    "red1": ([0, 120, 70], [10, 255, 255]),
    "red2": ([170, 120, 70], [180, 255, 255]),
    "blue": ([100, 150, 50], [140, 255, 255]),
    "green": ([40, 70, 70], [80, 255, 255])
}

# -------- STATE --------
busy = False
last_detected = None
stable_count = 0

# -------- SEND COMMAND --------
def send_command(cmd):
    global busy
    print(f"Sending: {cmd}")
    arduino.write((cmd + "\n").encode())
    busy = True

# -------- MAIN LOOP --------
while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Draw the Pick Zone target line
    cv2.line(frame, (PICK_X_CENTER - TOLERANCE, 0), (PICK_X_CENTER - TOLERANCE, frame.shape[0]), (0, 255, 255), 1)
    cv2.line(frame, (PICK_X_CENTER + TOLERANCE, 0), (PICK_X_CENTER + TOLERANCE, frame.shape[0]), (0, 255, 255), 1)
    cv2.putText(frame, "PICK ZONE", (PICK_X_CENTER - 40, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 2)

    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    detected_color = None
    in_pick_zone = False
    largest_area_overall = 0 # Track the absolute largest object on screen

    for color_name, (lower, upper) in color_ranges.items():
        if color_name == "red2":
            continue

        lower = np.array(lower)
        upper = np.array(upper)
        mask = cv2.inRange(hsv, lower, upper)

        if color_name == "red1":
            lower2, upper2 = color_ranges["red2"]
            mask2 = cv2.inRange(hsv, np.array(lower2), np.array(upper2))
            mask = mask | mask2
            color_name = "red"

        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        if contours:
            # IMPROVEMENT: Grab the single largest contour for this color
            largest_cnt = max(contours, key=cv2.contourArea)
            area = cv2.contourArea(largest_cnt)

            if area > MIN_AREA and area > largest_area_overall:
                largest_area_overall = area
                
                x, y, w, h = cv2.boundingRect(largest_cnt)
                cx = x + w // 2
                cy = y + h // 2
                
                # Draw bounding box
                cv2.rectangle(frame, (x, y), (x+w, y+h), (0,255,0), 2)
                cv2.circle(frame, (cx, cy), 5, (255, 0, 0), -1)
                cv2.putText(frame, color_name, (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,0), 2)

                # Check if it's in the zone
                if abs(cx - PICK_X_CENTER) <= TOLERANCE:
                    detected_color = color_name
                    in_pick_zone = True

    # -------- STABILITY & ZONE CHECK --------
    if detected_color == last_detected and in_pick_zone:
        stable_count += 1
    else:
        stable_count = 0

    last_detected = detected_color if in_pick_zone else None

    # -------- DECISION --------
    if (detected_color is not None and
        stable_count >= REQUIRED_STABLE and
        not busy):

        print(f"Object in zone, stable! Color: {detected_color}")

        # IMPROVEMENT: Continuous logic, no list depletion
        if detected_color in TARGET_COLORS:
            print("Target recognized → initiating pickup")
            send_command("PICK")
        else:
            print(f"Ignoring: {detected_color}")

        stable_count = 0 

    # -------- SERIAL FEEDBACK --------
    if arduino.in_waiting:
        msg = arduino.readline().decode().strip()
        print(f"Arduino: {msg}")

        if "ACTION: DONE" in msg:
            busy = False

    # -------- DISPLAY --------
    cv2.putText(frame, f"Targets: {TARGET_COLORS}", (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)
    cv2.imshow("System", frame)

    if cv2.waitKey(1) & 0xFF == 27:
        break

# -------- CLEANUP --------
cap.release()
cv2.destroyAllWindows()
arduino.close()

Waiting for Arduino to initialize...
Arduino: SYSTEM READY
Handshake complete. Starting camera...
Object in zone, stable! Color: red
Target recognized → initiating pickup
Sending: PICK
Arduino: ACK: PICK
Arduino: ACTION: START
Arduino: ACTION: DONE
Object in zone, stable! Color: blue
Target recognized → initiating pickup
Sending: PICK
Arduino: ACK: PICK
Arduino: ACTION: START
Arduino: ACTION: DONE
